# 04 · Common evaluation pipeline

Notebook 03 measured **one** configuration: plain CLIP. This notebook turns that into a **reusable
testing machine**: every mitigation method goes through exactly the same steps, so their numbers
can be compared fairly (Roberta's request).

```
queries + FairFace gallery ──► CLIP ──► [ method ] ──► ranking ──► metrics
                                         │                        ├─ bias: MaxSkew, MinSkew, NDKL
                                         │                        │        (uniform + gallery, k = 100 / 1000, vs chance)
                                         │                        ├─ utility: Flickr30K Recall@K
                                         │                        └─ cost: set-up time, latency, GPU memory, added params
              in-processing ── changes the model (Berg et al.)
              pre-processing ─ changes the embeddings before scoring (Chuang, Zhao, BendVLM)
              post-processing  changes the ranking after scoring (Kong et al.)
```

**What stays fixed for every method** (set once, in `configs/experiments/pipeline_v1.yaml`): the 364
queries, the gallery (Berg et al.'s protocol), the attributes, both reference distributions, the
cut-offs k, the chance level, and the Flickr30K utility set. **What changes:** only the method.

**Notebook 03 is unchanged.** This notebook first checks that the pipeline with *no* method gives
exactly notebook 03's numbers. Only then is it trusted for the mitigation methods.

## 0. Setup

Same as notebook 03. For a private repo, the `GITHUB_TOKEN` secret must be set (key icon, left sidebar).

In [ ]:
import os, sys

REPO = "simalyucel/clip-bias-mitigation"
ROOT = "/content/clip-bias-mitigation"

try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")      # only needed for a private repo
except Exception:
    token = None
url = f"https://{token}@github.com/{REPO}.git" if token else f"https://github.com/{REPO}.git"

if not os.path.exists(ROOT):
    !git clone -q {url} {ROOT}
else:
    !git -C {ROOT} pull -q
%cd {ROOT}
!pip -q install open_clip_torch datasets
!pip -q install -e .
sys.path.insert(0, f"{ROOT}/src")

In [ ]:
# must end with "43 passed" (36 old tests + 7 for the pipeline)
!python -m pytest -q

In [ ]:
import numpy as np, pandas as pd, torch
from pathlib import Path

from vlbias.utils.config import load_yaml
from vlbias.mitigation.base import Mitigation, NoMitigation, Compose, normalize
from vlbias.pipeline import EvaluationPipeline, compare, compare_costs, load_result

pd.set_option("display.precision", 3)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU)")

## 1. The protocol

`pipeline_v1.yaml` is a copy of `baseline_v1_berg.yaml` (notebook 03's setup) with three additions:

| Block | What it is for |
|---|---|
| `fit_data` | Labelled images a method may **learn** from (FairFace *train*, 20k images). A different split from the gallery that is searched, so no method is tested on its own training data. Only loaded if a method asks for it. |
| `targets` | Deliverable 2, Table 12: at least **5% less bias** and at most **5% less utility** than the baseline. The primary metrics (MaxSkew@1000, gallery; text→image R@5) are provisional. |
| `output_dir` | `results/pipeline_v1/<method>/`: one folder per method, same files in each. |

`USE_DRIVE = True` keeps the embedding cache across Colab sessions (recommended from now on: every
method reuses the same gallery embeddings).

In [ ]:
CONFIG = "configs/experiments/pipeline_v1.yaml"
USE_DRIVE = False

cfg = load_yaml(CONFIG)
CACHE_DIR = None
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    CACHE_DIR = "/content/drive/MyDrive/thesis/embeddings"
cfg

## 2. Build the pipeline (once)

This loads everything that does **not** depend on the method, once: the model, the queries, the
gallery and its labels, and the chance level (1,000 random rankings). Every method is then compared
against the **same** chance level. Takes a few minutes the first time (FairFace download, embeddings).

In [ ]:
pipe = EvaluationPipeline(cfg, cache_dir=CACHE_DIR)
print(len(pipe.queries), "queries |", pipe.gallery.name, len(pipe.gallery), "images |", pipe.attributes)

## 3. Run C0: no mitigation

`NoMitigation()` changes nothing, so this is plain CLIP, the reference every method is compared with.
`run` does, in order: (1) model, (2) set-up, (3) embedding transforms, (4) latency, (5) fairness,
(6) utility, and saves everything to `results/pipeline_v1/none/`.

In [ ]:
base = pipe.run(NoMitigation())
base.output_dir

## 4. Check: does the pipeline reproduce notebook 03?

The pipeline is only trustworthy if, with no mitigation, it gives **the same numbers** as notebook 03.
The repo contains notebook 03's results (`results/baseline_v1_berg_results.zip`); we compare every mean
in the summary.

**Expected:** a maximum difference of 0 (or ~1e-4 if the GPU encodes images very slightly differently
in a new session). Anything larger means something changed in the protocol: stop and check.

In [ ]:
import zipfile
with zipfile.ZipFile("results/baseline_v1_berg_results.zip") as z:
    nb03 = pd.read_csv(z.open("summary.csv"))

keys = ["query_set", "attribute", "reference", "k"]
cols = ["max_skew_mean", "min_skew_mean", "ndkl_mean", "share_max_skew_sig"]
m = nb03.merge(base.summary, on=keys, suffixes=("_nb03", "_pipe"))
diff = max((m[f"{c}_nb03"] - m[f"{c}_pipe"]).abs().max() for c in cols)
print(f"{len(m)} summary rows compared | largest difference: {diff:.2e}")
assert len(m) == len(nb03), "some rows are missing"
print("OK: pipeline == notebook 03" if diff < 1e-3 else "MISMATCH: check the config")

## 5. What the pipeline produces for every method

Three kinds of output, the same for every method:

1. **Bias** (`summary`): mean MaxSkew / MinSkew / NDKL over queries, the chance level and the share of
   significantly biased queries; per query set × attribute × reference × k. Uniform first, then gallery.
2. **Utility** (`utility`): Flickr30K Recall@1/5/10, both directions.
3. **Cost** (`costs`): see the table in section 5c.

In [ ]:
# 5a. Bias (Berg et al.'s prompts, k = 1000), uniform reference first, then gallery
cols = ["max_skew_mean", "max_skew_chance", "ndkl_mean", "share_max_skew_sig"]
s = base.summary.query("query_set == 'berg2022_test' and k == 1000").copy()
s["reference"] = pd.Categorical(s["reference"], ["uniform", "gallery"])
s.sort_values(["reference", "attribute"]).set_index(["reference", "attribute"])[cols]

In [ ]:
# 5b. Utility (should match notebook 03: text->image R@5 = 83.0)
base.utility.pivot(index="k", columns="direction", values="recall").round(1)

### 5c. Cost

| Metric | Meaning |
|---|---|
| `fit_seconds` | set-up of the method (e.g. computing a projection matrix, training prompts) |
| `fit_data_loading_seconds` | encoding the labelled fit images, if the method needs them (cached afterwards) |
| `transform_gallery_seconds`, `transform_queries_seconds` | applying the method to all embeddings |
| `latency_per_query_ms` | one search: transform the query + rank the whole gallery (top 1000) |
| `extra_parameters` | learned parameters the method adds (0 = training-free) |
| `gpu_peak_memory_mb` | peak GPU memory during the run |
| `fairness_eval_seconds`, `utility_eval_seconds` | time of the evaluation itself (not a cost of the method) |

For C0 all method costs are ~0: it is the reference.

In [ ]:
compare_costs([base]).T

## 6. How a method plugs in

A method is a small class with up to four hooks. It only writes the ones it needs; the rest default
to "change nothing":

| Hook | Category | Used by |
|---|---|---|
| `fit(ctx)` | set-up (any) | computing a projection matrix, training a classifier, ... |
| `adapt_model(vlm)` | **in-processing** | Berg et al. (returns their debiased model) |
| `transform_text(emb)` / `transform_image(emb)` | **pre-processing** | Chuang, Zhao, BendVLM |
| `rank(query_emb, gallery_emb, k)` | **post-processing** | Kong et al. |

`ctx` gives the method the model, a function to encode prompts (`ctx.encode_texts`) and, if needed,
labelled training embeddings (`ctx.fit_data()`). Methods go in `src/vlbias/mitigation/pre/`, `in_processing/`
or `post/`, following Roberta's categories.

**Combining methods** (one of Roberta's research directions) is built in:
`Compose([pre_method, post_method])` applies both and is evaluated like any other method.

The cell below is the template the next methods will follow (not run).

In [ ]:
class TemplateMethod(Mitigation):
    name = "template"       # results go to results/pipeline_v1/template/
    category = "pre"        # "pre", "in" or "post"

    def __init__(self, strength=1.0):
        self.strength = strength

    def fit(self, ctx):
        # e.g. encode prompts that describe the bias:
        # self.P = ...  ctx.encode_texts(["a photo of a man", "a photo of a woman"]) ...
        pass

    def transform_text(self, emb):
        return emb          # e.g. normalize(emb @ self.P.T)

    def params(self):
        return {"strength": self.strength}   # saved in method.yaml

# res = pipe.run(TemplateMethod())

## 7. The comparison table

`compare` puts all methods side by side for one reference and k: mean bias per attribute, Flickr30K
Recall@K, the **change vs C0 in %** (negative = less bias), and whether the Table 12 targets are met.
With only C0 it has one row per attribute; every new method adds rows.

To compare runs from **different sessions**, reload them from disk with
`load_result("results/pipeline_v1/<method>")`.

In [ ]:
t = cfg["targets"]
results = [base]            # later: [base, chuang, zhao, kong, ...]
for ref in ("uniform", "gallery"):
    display(compare(results, reference=ref, k=t["k"], query_set="berg2022_test", targets=t)
            .set_index(["method", "attribute"]).round(3))

## 8. Save

Zip `results/pipeline_v1/` (all methods run so far) and download it. Keep it outside the repo, named
after the date.

In [ ]:
out = Path(cfg["output_dir"])
!cd {out.parent} && zip -q -r /content/{out.name}_results.zip {out.name}
from google.colab import files
files.download(f"/content/{out.name}_results.zip")